In [ ]:
# === STANDART COLAB KURULUM HÜCRESİ ===
from google.colab import drive, userdata
drive.mount("/content/drive")

import os, sys
REPO_URL = "https://github.com/Oldzhay34/receipts-and-interactions.git"
BRANCH   = "main"
try:
    TOKEN = userdata.get("GH_TOKEN")
except Exception:
    TOKEN = None
url = REPO_URL.replace("https://", f"https://{TOKEN}@") if TOKEN else REPO_URL

if not os.path.exists("/content/repo"):
    !git clone -q -b {BRANCH} {url} /content/repo
else:
    !cd /content/repo && git fetch -q && git checkout -q {BRANCH} && git pull -q

%cd /content/repo
!pip install -q -r requirements.txt && pip install -q -e .

if "/content/repo/src" not in sys.path:
    sys.path.insert(0, "/content/repo/src")

os.environ["DATA_ROOT"] = "/content/drive/MyDrive/CME4403/data"
from recsys.config import load_config
CFG = load_config()
print("commit:", os.popen("git rev-parse --short HEAD").read().strip())

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

from recsys.io.artifact_store import ArtifactStore
from recsys.analysis.distribution import with_rating_stats
from recsys.features.binning import (
    EqualFrequencyBins,
    EqualWidthBins,
    binner_from_config,
    compare_binning_strategies,
    edges_for_manifest,
)
from recsys.viz.plots import bar_counts
from recsys.contracts import (
    BINNING_STAGE_SCHEMAS,
    LOADED_INTERACTIONS_SCHEMA,
    LOADED_RECIPES_SCHEMA,
    validate,
)

store = ArtifactStore.from_config(CFG)
loaded = store.load("loaded")

recipes = validate(loaded["recipes"], LOADED_RECIPES_SCHEMA)
interactions = validate(loaded["interactions"], LOADED_INTERACTIONS_SCHEMA)

BINNING_CFG = CFG.section("binning")  # config/config.yaml `binning` block: chosen bins

# Exploratory avg_rating (ratings > 0, rated recipes only); model targets: train-only in 1.9
recipes_rated = with_rating_stats(recipes, interactions, how="inner")
print(f"All recipes: {len(recipes):,} | Rated recipes: {len(recipes_rated):,}")

In [ ]:
minutes_strategies = {
    "domain": binner_from_config("minutes", BINNING_CFG["minutes"]),  # quick < 30 <= medium < 60
    "equal_frequency": EqualFrequencyBins(
        column="minutes",
        n_bins=3,
        labels=["quick", "medium", "long"],
        output_col="minutes_bin",
    ),
}
minutes_comp = compare_binning_strategies(recipes, "minutes", minutes_strategies)
display(minutes_comp)

In [ ]:
rating_domain = binner_from_config("avg_rating", BINNING_CFG["avg_rating"])  # 3.5 / 4.5
rating_eq_width = EqualWidthBins(
    column="avg_rating",
    n_bins=3,
    labels=["low", "medium", "high"],
    output_col="rating_class",
)
rating_eq_freq = EqualFrequencyBins(
    column="avg_rating",
    n_bins=3,
    labels=["low", "medium", "high"],
    output_col="rating_class",
)

rating_comp = compare_binning_strategies(
    recipes_rated,
    "avg_rating",
    {
        "domain_3.5_4.5": rating_domain,
        "equal_width": rating_eq_width,
        "equal_frequency": rating_eq_freq,
    },
)
display(rating_comp)

df_r_dom = rating_domain.fit_transform(recipes_rated)
df_r_freq = rating_eq_freq.fit_transform(recipes_rated)

fig_rating_dom = bar_counts(
    df_r_dom["rating_class"],
    top_n=5,
    title="avg_rating Classes — Domain Thresholds (3.5 / 4.5)",
    ylabel="Rating Class",
    horizontal=False,
)
fig_rating_freq = bar_counts(
    df_r_freq["rating_class"],
    top_n=5,
    title="avg_rating Classes — Equal Frequency",
    ylabel="Rating Class",
    horizontal=False,
)

In [ ]:
calories_strategies = {
    "equal_width": EqualWidthBins(
        column="calories",
        n_bins=3,
        labels=["low", "medium", "high"],
        output_col="calories_bin",
    ),
    "equal_frequency": EqualFrequencyBins(
        column="calories",
        n_bins=3,
        labels=["low", "medium", "high"],
        output_col="calories_bin",
    ),
}
calories_comp = compare_binning_strategies(recipes, "calories", calories_strategies)
display(calories_comp)

In [ ]:
saved = store.save(
    stage="binning",
    objects={
        "minutes_comparison": minutes_comp,
        "avg_rating_comparison": rating_comp,
        "calories_comparison": calories_comp,
        "fig_07_01_rating_domain": fig_rating_dom,
        "fig_07_02_rating_eq_freq": fig_rating_freq,
    },
    inputs=[loaded],
    schemas=BINNING_STAGE_SCHEMAS,  # tables validated before writing
    params={
        "minutes_domain_edges": edges_for_manifest(BINNING_CFG["minutes"]["edges"]),
        "avg_rating_domain_edges": edges_for_manifest(BINNING_CFG["avg_rating"]["edges"]),
        "n_bins": 3,
    },
)

plt.close(fig_rating_dom)
plt.close(fig_rating_freq)

print("Saved binning artifact:", saved.version)
print("\n=== MINUTES ===\n", minutes_comp.to_string(index=False))
print("\n=== AVG_RATING ===\n", rating_comp.to_string(index=False))
print("\n=== CALORIES ===\n", calories_comp.to_string(index=False))